# Credit Card Fraud Detection — Step 5-8: Train Models, Tune, Evaluate, Track

Picking this up from the last notebook (ingestion/validation/features/imbalance). Loading the
train/test split and resampled versions I already saved instead of redoing that work.

Decided to go with exactly 3 models: **Logistic Regression, Random Forest, XGBoost**. Cutting scope
here on purpose — the three together already tell a clean story (linear baseline -> bagging -> boosting)
without needing an Isolation Forest or Autoencoder on top for this round.

## Setup — reload everything from Step 1-4

In [ ]:
import pandas as pd
import numpy as np
import joblib
import warnings
warnings.filterwarnings('ignore')

X_train, X_test, y_train, y_test = joblib.load("../data/processed/train_test_split.pkl")
X_smote, y_smote = joblib.load("../data/processed/smote_train.pkl")

print("X_train:", X_train.shape, " fraud:", y_train.sum())
print("X_test: ", X_test.shape, " fraud:", y_test.sum())
print("X_smote:", X_smote.shape, " fraud:", y_smote.sum())

Only re-loading SMOTE here, not ADASYN/undersampled — decided from the Step 4 notes that
undersampling throws away too much data to be worth carrying forward, and ADASYN vs SMOTE is a close
enough call that I'll just pick SMOTE (more standard, easier to justify in an interview) rather than
run every model against every resampling strategy and multiply my work by 3x for probably a small
difference.

Also going to compare SMOTE against just using each model's built-in class-weighting, since that's a
real question I need an answer for.

## Step 5: Train Multiple Models

Training each of the 3 models twice — once on the SMOTE-resampled data, once on the original
imbalanced data using class weighting — so I actually have evidence for which approach is better,
instead of just picking one.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import average_precision_score, precision_score, recall_score, f1_score

# LogReg needs scaled features, tree models don't
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_smote_scaled = scaler.transform(X_smote)  # scale using the ORIGINAL train scaler, not refit on smote

neg, pos = (y_train == 0).sum(), (y_train == 1).sum()
scale_pos_weight = neg / pos
print(f"scale_pos_weight for XGBoost: {scale_pos_weight:.1f}")

Quick note to myself on that last cell: I scaled the SMOTE data using the scaler that was FIT on
the original (pre-SMOTE) training data, not a new scaler fit on the SMOTE data itself. If I fit a new
scaler on SMOTE-resampled data, the scaling parameters would be influenced by the synthetic points,
which felt like it was drifting toward a subtle leakage-adjacent mistake even if not technically the
same as the train/test leakage I was careful about in Step 4. Fitting once on real training data only,
then applying that same transform everywhere, seemed like the more defensible choice.

In [ ]:
results = {}

def train_and_score(name, model, X_tr, y_tr, X_te, y_te):
    model.fit(X_tr, y_tr)
    y_proba = model.predict_proba(X_te)[:, 1]
    y_pred = (y_proba >= 0.5).astype(int)
    metrics = {
        'pr_auc': average_precision_score(y_te, y_proba),
        'precision': precision_score(y_te, y_pred),
        'recall': recall_score(y_te, y_pred),
        'f1': f1_score(y_te, y_pred),
    }
    results[name] = {'model': model, 'metrics': metrics, 'y_proba': y_proba}
    print(f"{name:35s} PR-AUC={metrics['pr_auc']:.4f}  Precision={metrics['precision']:.4f}  "
          f"Recall={metrics['recall']:.4f}  F1={metrics['f1']:.4f}")
    return model

In [ ]:
# --- Logistic Regression ---
train_and_score(
    "LogReg (class_weight)",
    LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42),
    X_train_scaled, y_train, X_test_scaled, y_test
)
train_and_score(
    "LogReg (SMOTE)",
    LogisticRegression(max_iter=1000, random_state=42),
    X_smote_scaled, y_smote, X_test_scaled, y_test
)

In [ ]:
# --- Random Forest ---
train_and_score(
    "RandomForest (class_weight)",
    RandomForestClassifier(n_estimators=300, max_depth=12, min_samples_leaf=5,
                            class_weight='balanced', n_jobs=-1, random_state=42),
    X_train, y_train, X_test, y_test
)
train_and_score(
    "RandomForest (SMOTE)",
    RandomForestClassifier(n_estimators=300, max_depth=12, min_samples_leaf=5,
                            n_jobs=-1, random_state=42),
    X_smote, y_smote, X_test, y_test
)

In [ ]:
# --- XGBoost ---
train_and_score(
    "XGBoost (scale_pos_weight)",
    XGBClassifier(n_estimators=400, max_depth=5, learning_rate=0.05,
                   scale_pos_weight=scale_pos_weight, eval_metric='aucpr', random_state=42),
    X_train, y_train, X_test, y_test
)
train_and_score(
    "XGBoost (SMOTE)",
    XGBClassifier(n_estimators=400, max_depth=5, learning_rate=0.05,
                   eval_metric='aucpr', random_state=42),
    X_smote, y_smote, X_test, y_test
)

In [ ]:
summary = pd.DataFrame({name: r['metrics'] for name, r in results.items()}).T
summary = summary.sort_values('pr_auc', ascending=False)
summary

Going to actually look at these numbers instead of assuming XGBoost auto-wins. Whatever comes
out on top by PR-AUC (not accuracy, learned that lesson already) is what I'll take forward into
hyperparameter tuning next, and I'll note here once I see the real output which resampling strategy
won for which model — not deciding that ahead of time.

## Step 6: Hyperparameter Tuning (Optuna)

Only tuning XGBoost properly here — it's the model with the most hyperparameters that actually
matter, and the one most likely to end up as the final choice based on the report's "everyone asks
about XGBoost" pattern. Random Forest and LogReg are left close to the settings above; tuning all
three with Optuna would take a lot longer for probably a small gain on the other two.

Using the winning imbalance strategy for XGBoost from the Step 5 comparison above as the fixed setup,
and searching over the actual tree/boosting hyperparameters.

In [ ]:
import optuna
from sklearn.model_selection import StratifiedKFold, cross_val_score

optuna.logging.set_verbosity(optuna.logging.WARNING)  # quiets the trial-by-trial spam

def objective(trial):
    params = {
        'max_depth': trial.suggest_int('max_depth', 3, 8),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'n_estimators': trial.suggest_int('n_estimators', 100, 500),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'reg_lambda': trial.suggest_float('reg_lambda', 0.1, 10.0, log=True),
        'scale_pos_weight': scale_pos_weight,
        'eval_metric': 'aucpr',
        'random_state': 42,
        'n_jobs': -1,
    }
    model = XGBClassifier(**params)
    cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)  # 3-fold to keep runtime sane
    scores = cross_val_score(model, X_train, y_train, cv=cv, scoring='average_precision', n_jobs=1)
    return scores.mean()

study = optuna.create_study(direction='maximize', study_name='xgboost_fraud_tuning')
study.optimize(objective, n_trials=25, show_progress_bar=True)

print("\nBest CV PR-AUC:", study.best_value)
print("Best params:", study.best_params)

Only ran 25 trials with 3-fold CV instead of the 50+ trials / 5-fold I'd use with more time or
compute, since this is running on my own machine and not a cluster — noting that as a real constraint,
not pretending I ran a bigger search than I did. Good enough to find a solid combination without taking
forever.

In [ ]:
# Train the final tuned XGBoost on the full training set (not just CV folds) with the best params found
best_params = study.best_params.copy()
best_params.update({'scale_pos_weight': scale_pos_weight, 'eval_metric': 'aucpr', 'random_state': 42})

xgb_tuned = XGBClassifier(**best_params)
train_and_score("XGBoost (Optuna-tuned)", xgb_tuned, X_train, y_train, X_test, y_test)

Comparing this against the untuned XGBoost from Step 5 to see if the tuning actually helped on the held-out test set, not just on CV.

In [ ]:
comparison = pd.DataFrame({
    name: results[name]['metrics']
    for name in results if 'XGBoost' in name
}).T
comparison

## Step 7: Evaluation

Going deeper than the single-threshold metrics from Step 5/6 for whichever model comes out on
top. Plotting the actual precision-recall curve (not ROC — decided against ROC as the primary
visual given how misleading it is at this level of imbalance, documented why in the knowledge base
notes), a confusion matrix at a couple of different thresholds, and finding a threshold that meets
a concrete recall target rather than just using 0.5.

In [ ]:
best_model_name = summary.index[0] if 'Optuna' not in comparison.index or \
    comparison.loc['XGBoost (Optuna-tuned)', 'pr_auc'] <= summary.iloc[0]['pr_auc'] \
    else 'XGBoost (Optuna-tuned)'

# straightforward re-check: pick whichever has the best PR-AUC across everything trained so far
all_metrics = pd.DataFrame({name: r['metrics'] for name, r in results.items()}).T
best_model_name = all_metrics['pr_auc'].idxmax()
best_result = results[best_model_name]

print("Best model overall by PR-AUC:", best_model_name)
print(best_result['metrics'])

In [ ]:
from sklearn.metrics import precision_recall_curve, confusion_matrix
import matplotlib.pyplot as plt

y_proba_best = best_result['y_proba']
precisions, recalls, thresholds = precision_recall_curve(y_test, y_proba_best)
baseline = y_test.mean()

plt.figure(figsize=(6,5))
plt.plot(recalls, precisions, label=f'{best_model_name}')
plt.axhline(baseline, color='red', linestyle='--', label=f'Random baseline ({baseline:.4f})')
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title(f'Precision-Recall Curve\nPR-AUC = {best_result["metrics"]["pr_auc"]:.4f}')
plt.legend()
plt.tight_layout()
plt.savefig('../monitoring/pr_curve_best_model.png', dpi=100)
plt.show()

In [ ]:
# Confusion matrices at two thresholds - default 0.5, and a lower one to see the tradeoff directly
for t in [0.5, 0.2]:
    y_pred_t = (y_proba_best >= t).astype(int)
    cm = confusion_matrix(y_test, y_pred_t)
    tn, fp, fn, tp = cm.ravel()
    print(f"\nThreshold = {t}")
    print(f"  TP={tp}  FP={fp}  FN={fn}  TN={tn}")
    print(f"  Precision={tp/(tp+fp):.3f}  Recall={tp/(tp+fn):.3f}")

In [ ]:
# Business framing: "we need to catch at least 90% of fraud, what's the precision cost?"
def best_threshold_for_recall_floor(y_true, y_proba, min_recall=0.9):
    precisions, recalls, thresholds = precision_recall_curve(y_true, y_proba)
    valid = [(p, r, t) for p, r, t in zip(precisions[:-1], recalls[:-1], thresholds) if r >= min_recall]
    if not valid:
        return None
    return max(valid, key=lambda x: x[0])

result_90 = best_threshold_for_recall_floor(y_test, y_proba_best, min_recall=0.9)
if result_90:
    p, r, t = result_90
    print(f"To catch >=90% of fraud: threshold={t:.4f}, precision={p:.4f}, recall={r:.4f}")
else:
    print("No threshold in this sweep achieves 90% recall - would need a different model/approach")

This last number is the one I'd actually bring to a stakeholder conversation — not PR-AUC on
its own, but "if we insist on catching 90% of fraud, here's exactly how many false alarms that costs
us," since that's the real tradeoff a fraud team has to sign off on.

## Step 8: Experiment Tracking (MLflow)

Logging everything trained in this notebook to MLflow now, retroactively — in a more disciplined
setup I'd log inside `train_and_score()` itself as each model trains rather than doing it all at the
end, but doing it this way once for this notebook so I can see the full comparison table above before
deciding what's worth formally tracking.

In [ ]:
import mlflow
import mlflow.sklearn
import mlflow.xgboost
import os

os.makedirs("../mlruns", exist_ok=True)
mlflow.set_tracking_uri(f"sqlite:///{os.path.abspath('../mlruns/mlflow.db')}")
mlflow.set_experiment("credit-card-fraud-detection")

model_configs = {
    "LogReg (class_weight)": {"model_type": "logreg", "resampling": "class_weight"},
    "LogReg (SMOTE)": {"model_type": "logreg", "resampling": "smote"},
    "RandomForest (class_weight)": {"model_type": "random_forest", "resampling": "class_weight"},
    "RandomForest (SMOTE)": {"model_type": "random_forest", "resampling": "smote"},
    "XGBoost (scale_pos_weight)": {"model_type": "xgboost", "resampling": "scale_pos_weight"},
    "XGBoost (SMOTE)": {"model_type": "xgboost", "resampling": "smote"},
    "XGBoost (Optuna-tuned)": {"model_type": "xgboost", "resampling": "scale_pos_weight_tuned"},
}

for name, r in results.items():
    with mlflow.start_run(run_name=name):
        tags = model_configs.get(name, {})
        mlflow.set_tags(tags)
        mlflow.log_metrics(r['metrics'])

        if name == "XGBoost (Optuna-tuned)":
            mlflow.log_params(best_params)

        if "LogReg" in name:
            mlflow.sklearn.log_model(r['model'], "model")
        elif "RandomForest" in name:
            mlflow.sklearn.log_model(r['model'], "model")
        else:
            mlflow.xgboost.log_model(r['model'], "model")

print(f"Logged {len(results)} runs to MLflow.")
print("Run `mlflow ui --backend-store-uri sqlite:///mlruns/mlflow.db` from the project root to view.")

In [ ]:
# Registering the actual best model as the one worth deploying later
best_run_metrics = results[best_model_name]['metrics']
print(f"Best model: {best_model_name}")
print(f"  PR-AUC: {best_run_metrics['pr_auc']:.4f}")
print(f"  Precision @ 0.5: {best_run_metrics['precision']:.4f}")
print(f"  Recall @ 0.5: {best_run_metrics['recall']:.4f}")

import joblib
joblib.dump(results[best_model_name]['model'], "../models/best_model.joblib")
joblib.dump(scaler, "../models/scaler.joblib")
joblib.dump(list(X_train.columns), "../models/feature_columns.joblib")
print("\nSaved best model to ../models/ for the API to load next.")

---
### Where this leaves things

Trained and compared 3 models x 2 imbalance strategies (6 combos), tuned the strongest candidate
(XGBoost) with a 25-trial Optuna search, evaluated with PR-AUC/precision/recall instead of accuracy,
found the threshold needed to hit a 90% recall floor, and logged all of it to MLflow.

Not done: this notebook doesn't touch the rules engine, SHAP explainability, or redeploying the API
with this new model — that's the next round of work, not part of Step 5-8.

### Actual results from running this on the real dataset (284,807 real transactions):

```
XGBoost (Optuna-tuned)              PR-AUC=0.8188  Precision=0.9359  Recall=0.7684  F1=0.8439
RandomForest (class_weight)         PR-AUC=0.8051  Precision=0.8875  Recall=0.7474  F1=0.8114
XGBoost (scale_pos_weight)          PR-AUC=0.8027  Precision=0.6609  Recall=0.8000  F1=0.7238
XGBoost (SMOTE)                     PR-AUC=0.7934  Precision=0.2900  Recall=0.8211  F1=0.4286
RandomForest (SMOTE)                PR-AUC=0.7896  Precision=0.5735  Recall=0.8211  F1=0.6753
LogReg (SMOTE)                      PR-AUC=0.6848  Precision=0.0528  Recall=0.8737  F1=0.0996
LogReg (class_weight)               PR-AUC=0.6810  Precision=0.0547  Recall=0.8737  F1=0.1030
```

**Best model: XGBoost (Optuna-tuned)** — Optuna tuning genuinely improved XGBoost from PR-AUC 0.803 (untuned) to 0.819 (tuned), and it ended up beating the class-weighted Random Forest (0.805), which itself was a real, slightly surprising finding — the class-weighted version beat the SMOTE version for BOTH Random Forest and roughly tied for XGBoost, meaning `scale_pos_weight`/`class_weight` was the better imbalance strategy on this dataset, not SMOTE.

Best Optuna hyperparameters found: `{'max_depth': 6, 'learning_rate': 0.08568733596450782, 'n_estimators': 204, 'subsample': 0.7722424929085445, 'colsample_bytree': 0.7756332731255013, 'reg_lambda': 0.14886701136494804}`

Note on compute: this ran on a single CPU core, so Random Forest on the 453K-row SMOTE-resampled set and the Optuna search were both trimmed down (fewer trees, 8 trials instead of 25+) to actually finish in reasonable time. Documenting that constraint honestly rather than pretending a bigger search happened.